# PoC RAG Agentico para Sector Mineria (Capstone 1)

Este notebook implementa una Prueba de Concepto de un sistema RAG (Retrieval-Augmented Generation)
Agentico que procesa documentos tecnicos de ingenieria/mineria y permite hacer consultas sobre ellos,
utilizando Function Calling nativo de Google Gemini.

## PASO 1: Instalacion de Dependencias y Configuracion

In [1]:
# Instalacion de dependencias desde requirements.txt
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])
print("Dependencias instaladas correctamente.")

Dependencias instaladas correctamente.


In [2]:
# Carga de variables de entorno y configuracion del SDK de Google
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise EnvironmentError("No se encontro la variable GOOGLE_API_KEY en el archivo .env")

# Inicializar el cliente del SDK de Google GenAI
client = genai.Client(api_key=api_key)

print("SDK de Google GenAI configurado correctamente.")
print(f"API Key cargada: {api_key[:10]}...{api_key[-4:]}")

SDK de Google GenAI configurado correctamente.
API Key cargada: AIzaSyDRTe...0SYQ


In [3]:
# Constantes del proyecto
# Se intenta con gemini-2.5-flash primero (modelo estable mas reciente),
# luego modelos alternativos en caso de no estar disponibles.
MODELO_LLM = "gemini-2.5-flash"
MODELO_EMBEDDING = "gemini-embedding-001"
DIRECTORIO_DATOS = "data"
NOMBRE_COLECCION = "mineria_tecnica"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 3

print(f"Modelo LLM: {MODELO_LLM}")
print(f"Modelo Embedding: {MODELO_EMBEDDING}")
print(f"Directorio de datos: {DIRECTORIO_DATOS}")
print(f"Chunk size: {CHUNK_SIZE} | Overlap: {CHUNK_OVERLAP}")
print(f"Top-K para recuperacion: {TOP_K}")

Modelo LLM: gemini-2.5-flash
Modelo Embedding: gemini-embedding-001
Directorio de datos: data
Chunk size: 1000 | Overlap: 200
Top-K para recuperacion: 3


## PASO 2: Ingesta de Datos (Document Ingestion)

In [4]:
import os
from typing import Optional
from pypdf import PdfReader


def load_pdfs(directory_path: str) -> list[dict]:
    """Lee iterativamente los archivos PDF dentro del directorio indicado.

    Extrae el texto crudo de cada pagina y conserva metadatos basicos
    como el nombre del archivo y el numero de pagina.

    Args:
        directory_path: Ruta al directorio que contiene los archivos PDF.

    Returns:
        Lista de diccionarios con las claves 'text' y 'metadata'.
    """
    documentos: list[dict] = []

    if not os.path.isdir(directory_path):
        print(f"El directorio '{directory_path}' no existe.")
        return documentos

    archivos_pdf = [f for f in sorted(os.listdir(directory_path)) if f.lower().endswith(".pdf")]

    if not archivos_pdf:
        print(f"No se encontraron archivos PDF en '{directory_path}'.")
        return documentos

    for nombre_archivo in archivos_pdf:
        ruta_completa = os.path.join(directory_path, nombre_archivo)
        print(f"Procesando: {nombre_archivo}")
        try:
            reader = PdfReader(ruta_completa)
            for num_pagina, pagina in enumerate(reader.pages, start=1):
                texto = pagina.extract_text()
                if texto and texto.strip():
                    documentos.append({
                        "text": texto,
                        "metadata": {
                            "nombre_archivo": nombre_archivo,
                            "numero_pagina": num_pagina,
                            "total_paginas": len(reader.pages),
                        }
                    })
        except Exception as e:
            print(f"Error al leer '{nombre_archivo}': {e}")
            continue

    print(f"\nIngesta completada: {len(documentos)} paginas extraidas de {len(archivos_pdf)} archivo(s).")
    return documentos


# Ejecutar ingesta
documentos_crudos = load_pdfs(DIRECTORIO_DATOS)

# Vista previa
if documentos_crudos:
    ejemplo = documentos_crudos[0]
    print(f"\nEjemplo de documento:")
    print(f"  Archivo: {ejemplo['metadata']['nombre_archivo']}")
    print(f"  Pagina: {ejemplo['metadata']['numero_pagina']}")
    print(f"  Longitud del texto: {len(ejemplo['text'])} caracteres")
    print(f"  Primeros 300 chars: {ejemplo['text'][:300]}...")

Procesando: Resumen-Ejecutivo-de-la-Tercera-MEIA-Las-Bambas.pdf
Procesando: rpm-2024.pdf
Procesando: rsso-2024.pdf

Ingesta completada: 689 paginas extraidas de 3 archivo(s).

Ejemplo de documento:
  Archivo: Resumen-Ejecutivo-de-la-Tercera-MEIA-Las-Bambas.pdf
  Pagina: 1
  Longitud del texto: 198 caracteres
  Primeros 300 chars: © 2016 SNC-Lavalin Perú 
Derechos reservados - Confidencial 
SECCIÓN 1 
Resumen Ejecutivo de la Tercera 
Modificación del Estudio de Impacto 
Ambiental de la Unidad Minera Las 
Bambas 
000058
000058...


## PASO 3: Segmentacion de Texto (Chunking)

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def segmentar_documentos(
    documentos: list[dict],
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> list[dict]:
    """Segmenta los documentos en chunks mas pequenos usando RecursiveCharacterTextSplitter.

    Mantiene los metadatos originales y agrega el indice del chunk dentro
    de cada pagina.

    Args:
        documentos: Lista de diccionarios provenientes de load_pdfs.
        chunk_size: Tamano maximo de cada chunk en caracteres.
        chunk_overlap: Cantidad de caracteres de solapamiento entre chunks.

    Returns:
        Lista de diccionarios con 'text' y 'metadata' por cada chunk.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks: list[dict] = []

    for doc in documentos:
        fragmentos = splitter.split_text(doc["text"])
        for idx, fragmento in enumerate(fragmentos):
            chunks.append({
                "text": fragmento,
                "metadata": {
                    **doc["metadata"],
                    "chunk_index": idx,
                },
            })

    print(f"Segmentacion completada: {len(chunks)} chunks generados a partir de {len(documentos)} paginas.")
    return chunks


# Ejecutar segmentacion
chunks = segmentar_documentos(documentos_crudos, CHUNK_SIZE, CHUNK_OVERLAP)

# Vista previa
if chunks:
    ejemplo_chunk = chunks[0]
    print(f"\nEjemplo de chunk:")
    print(f"  Archivo: {ejemplo_chunk['metadata']['nombre_archivo']}")
    print(f"  Pagina: {ejemplo_chunk['metadata']['numero_pagina']}")
    print(f"  Chunk index: {ejemplo_chunk['metadata']['chunk_index']}")
    print(f"  Longitud: {len(ejemplo_chunk['text'])} caracteres")
    print(f"  Texto: {ejemplo_chunk['text'][:200]}...")

Segmentacion completada: 1991 chunks generados a partir de 689 paginas.

Ejemplo de chunk:
  Archivo: Resumen-Ejecutivo-de-la-Tercera-MEIA-Las-Bambas.pdf
  Pagina: 1
  Chunk index: 0
  Longitud: 198 caracteres
  Texto: © 2016 SNC-Lavalin Perú 
Derechos reservados - Confidencial 
SECCIÓN 1 
Resumen Ejecutivo de la Tercera 
Modificación del Estudio de Impacto 
Ambiental de la Unidad Minera Las 
Bambas 
000058
000058...


## PASO 4: Generacion de Embeddings y Vector Store

In [6]:
import time
from typing import Optional


def generar_embeddings(
    textos: list[str],
    modelo: str = "gemini-embedding-001",
    batch_size: int = 100,
) -> list[list[float]]:
    """Genera embeddings para una lista de textos usando la API de Google.

    Divide los textos en lotes para respetar los limites de la API y agrega
    reintentos basicos con backoff exponencial.

    Args:
        textos: Lista de strings a vectorizar.
        modelo: Nombre del modelo de embeddings a utilizar.
        batch_size: Tamano de cada lote enviado a la API.

    Returns:
        Lista de vectores (listas de floats) correspondientes a cada texto.
    """
    todos_los_embeddings: list[list[float]] = []
    total_lotes = (len(textos) + batch_size - 1) // batch_size

    for i in range(0, len(textos), batch_size):
        lote = textos[i : i + batch_size]
        numero_lote = (i // batch_size) + 1
        print(f"  Generando embeddings para lote {numero_lote}/{total_lotes} ({len(lote)} textos)...")

        max_reintentos = 3
        for intento in range(max_reintentos):
            try:
                resultado = client.models.embed_content(
                    model=modelo,
                    contents=lote,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
                )
                embeddings_lote = [e.values for e in resultado.embeddings]
                todos_los_embeddings.extend(embeddings_lote)
                break
            except Exception as e:
                if intento < max_reintentos - 1:
                    espera = 2 ** intento
                    print(f"    Error en lote {numero_lote} (intento {intento + 1}): {e}")
                    print(f"    Reintentando en {espera}s...")
                    time.sleep(espera)
                else:
                    print(f"    Error definitivo en lote {numero_lote}: {e}")
                    raise

    print(f"Embeddings generados: {len(todos_los_embeddings)} vectores.")
    return todos_los_embeddings


# Extraer solo los textos para la generacion de embeddings
textos_chunks = [c["text"] for c in chunks]

print("Iniciando generacion de embeddings...")
embeddings = generar_embeddings(textos_chunks, MODELO_EMBEDDING)
print(f"Dimension de cada embedding: {len(embeddings[0])}")

Iniciando generacion de embeddings...
  Generando embeddings para lote 1/20 (100 textos)...
  Generando embeddings para lote 2/20 (100 textos)...
  Generando embeddings para lote 3/20 (100 textos)...
  Generando embeddings para lote 4/20 (100 textos)...
  Generando embeddings para lote 5/20 (100 textos)...
  Generando embeddings para lote 6/20 (100 textos)...
  Generando embeddings para lote 7/20 (100 textos)...
  Generando embeddings para lote 8/20 (100 textos)...
  Generando embeddings para lote 9/20 (100 textos)...
  Generando embeddings para lote 10/20 (100 textos)...
  Generando embeddings para lote 11/20 (100 textos)...
  Generando embeddings para lote 12/20 (100 textos)...
  Generando embeddings para lote 13/20 (100 textos)...
  Generando embeddings para lote 14/20 (100 textos)...
  Generando embeddings para lote 15/20 (100 textos)...
  Generando embeddings para lote 16/20 (100 textos)...
  Generando embeddings para lote 17/20 (100 textos)...
  Generando embeddings para lote 18/

In [7]:
import chromadb

# Inicializar ChromaDB en memoria
chroma_client = chromadb.Client()

# Crear (o recrear) la coleccion
try:
    chroma_client.delete_collection(name=NOMBRE_COLECCION)
except Exception:
    pass

coleccion = chroma_client.create_collection(
    name=NOMBRE_COLECCION,
    metadata={"hnsw:space": "cosine"},
)

# Preparar IDs, metadatos y documentos para ChromaDB
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = []
for c in chunks:
    meta = {}
    for k, v in c["metadata"].items():
        meta[k] = str(v) if not isinstance(v, (str, int, float, bool)) else v
    metadatas.append(meta)

# Insertar en lotes de 500 (limite de ChromaDB)
BATCH_CHROMA = 500
for i in range(0, len(ids), BATCH_CHROMA):
    fin = min(i + BATCH_CHROMA, len(ids))
    coleccion.add(
        ids=ids[i:fin],
        embeddings=embeddings[i:fin],
        documents=textos_chunks[i:fin],
        metadatas=metadatas[i:fin],
    )

print(f"\nVector Store creado exitosamente.")
print(f"  Coleccion: '{NOMBRE_COLECCION}'")
print(f"  Total de documentos indexados: {coleccion.count()}")


Vector Store creado exitosamente.
  Coleccion: 'mineria_tecnica'
  Total de documentos indexados: 1991


## PASO 5: Creacion de la Herramienta de Recuperacion (Retriever Tool)

In [8]:
def buscar_informacion_tecnica(query: str) -> str:
    """Busca informacion tecnica relevante en los documentos de mineria indexados.

    Recibe una consulta en lenguaje natural, la convierte en un embedding,
    realiza una busqueda de similitud en ChromaDB y retorna los fragmentos
    mas relevantes junto con la referencia al documento de origen.

    Args:
        query: Consulta en lenguaje natural del usuario.

    Returns:
        String con los fragmentos relevantes encontrados, incluyendo
        la citacion del documento de origen (nombre de archivo y pagina).
    """
    try:
        # Generar embedding de la consulta
        resultado_emb = client.models.embed_content(
            model=MODELO_EMBEDDING,
            contents=query,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
        )
        query_embedding = resultado_emb.embeddings[0].values

        # Buscar en ChromaDB los Top-K chunks mas similares
        resultados = coleccion.query(
            query_embeddings=[query_embedding],
            n_results=TOP_K,
        )

        if not resultados["documents"] or not resultados["documents"][0]:
            return "No se encontraron documentos relevantes para la consulta."

        # Formatear resultados con citacion de origen
        fragmentos = []
        for i, (doc, meta) in enumerate(
            zip(resultados["documents"][0], resultados["metadatas"][0]),
            start=1,
        ):
            nombre = meta.get("nombre_archivo", "Desconocido")
            pagina = meta.get("numero_pagina", "N/A")
            fragmento = (
                f"Fuente {i}: [{nombre}, pagina {pagina}]\n"
                f"{doc}\n"
            )
            fragmentos.append(fragmento)

        return "\n".join(fragmentos)

    except Exception as e:
        return f"Error al realizar la busqueda: {e}"


# Probar la funcion de recuperacion directamente
print("Prueba directa de la herramienta de recuperacion:")
print("=" * 60)
resultado_prueba = buscar_informacion_tecnica("seguridad en mineria")
print(resultado_prueba[:500])
print("...")

Prueba directa de la herramienta de recuperacion:
Fuente 1: [rsso-2024.pdf, pagina 296]
GUÍAS
REGLAMENTO
DE SEGURIDAD Y SALUD OCUPACIONAL EN MINERÍA

Fuente 2: [rsso-2024.pdf, pagina 180]
Ministerio de Energía y Minas 
180
Ar culo 
ART. 379.-  Se instalará sistemas de protección contra vuelcos en: 
tractores y cargadores frontales de orugas, motoniveladoras , cargadores y 
tractores de llantas. Su instalación debe hacerse de confo rmidad con las 
recomendaciones del fabricante. 
CAPÍTULO VI
EDIFICACIONES E INSTALACIONES
SUBCAPÍTULO I 
EDIFICAC
...


In [9]:
# Declaracion de la herramienta (Tool) para Gemini Function Calling
buscar_informacion_tecnica_declaration = {
    "name": "buscar_informacion_tecnica",
    "description": (
        "Busca informacion tecnica relevante en la base de documentos de mineria "
        "indexados. Utiliza esta herramienta para responder cualquier consulta "
        "sobre reglamentos de seguridad, normas mineras, procedimientos tecnicos, "
        "o cualquier tema cubierto por los documentos cargados."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "Consulta en lenguaje natural sobre temas tecnicos de mineria. "
                    "Ejemplo: 'requisitos de seguridad para trabajos en altura'"
                ),
            }
        },
        "required": ["query"],
    },
}

# Mapeo de funciones disponibles
funciones_disponibles = {
    "buscar_informacion_tecnica": buscar_informacion_tecnica,
}

print("Herramienta de recuperacion declarada correctamente para Function Calling.")
print(f"Nombre: {buscar_informacion_tecnica_declaration['name']}")

Herramienta de recuperacion declarada correctamente para Function Calling.
Nombre: buscar_informacion_tecnica


## PASO 6: Instanciacion del Agente (Google Gemini) y Prueba

In [10]:
# Configuracion del agente con Function Calling

SYSTEM_INSTRUCTION = (
    "Eres un Agente Especialista en Ingenieria Minera. "
    "Tu tarea es responder consultas tecnicas utilizando UNICAMENTE "
    "la herramienta de recuperacion de informacion provista. "
    "Si la informacion no esta en los documentos, debes indicar que no lo sabes. "
    "Cita siempre el documento de origen. "
    "Responde en espanol."
)

# Configurar Tools y GenerateContentConfig
tools = types.Tool(function_declarations=[buscar_informacion_tecnica_declaration])
config_agente = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[tools],
    temperature=0.2,
)

print("Agente configurado.")
print(f"Modelo: {MODELO_LLM}")
print(f"System Instruction: {SYSTEM_INSTRUCTION[:80]}...")

Agente configurado.
Modelo: gemini-2.5-flash
System Instruction: Eres un Agente Especialista en Ingenieria Minera. Tu tarea es responder consulta...


In [11]:
def ejecutar_consulta_agente(pregunta: str) -> str:
    """Ejecuta una consulta completa con el agente, manejando el ciclo
    de Function Calling manualmente.

    El flujo es:
    1. Enviar la pregunta al modelo con las Tools declaradas.
    2. Si el modelo responde con un function_call, ejecutar la funcion localmente.
    3. Enviar el resultado de la funcion de vuelta al modelo.
    4. El modelo genera la respuesta final incorporando la informacion recuperada.

    Args:
        pregunta: Consulta del usuario en lenguaje natural.

    Returns:
        Respuesta final del agente como string.
    """
    print(f"Pregunta: {pregunta}")
    print("-" * 60)

    # Paso 1: enviar la pregunta al modelo
    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=pregunta)],
        )
    ]

    response = client.models.generate_content(
        model=MODELO_LLM,
        contents=contents,
        config=config_agente,
    )

    # Paso 2: verificar si el modelo solicita un function call
    max_iteraciones = 5
    iteracion = 0

    while iteracion < max_iteraciones:
        iteracion += 1

        # Buscar si hay un function_call en la respuesta
        function_call = None
        for part in response.candidates[0].content.parts:
            if part.function_call:
                function_call = part.function_call
                break

        if not function_call:
            # No hay function call, el modelo dio una respuesta directa
            break

        # Ejecutar la funcion localmente
        nombre_funcion = function_call.name
        argumentos = dict(function_call.args)
        function_id = function_call.id

        print(f"  [Iteracion {iteracion}] El modelo solicita: {nombre_funcion}({argumentos})")

        if nombre_funcion in funciones_disponibles:
            resultado_funcion = funciones_disponibles[nombre_funcion](**argumentos)
            print(f"  [Iteracion {iteracion}] Funcion ejecutada. Resultado: {len(resultado_funcion)} chars")
        else:
            resultado_funcion = f"Error: La funcion '{nombre_funcion}' no esta disponible."
            print(f"  [Iteracion {iteracion}] Funcion no encontrada: {nombre_funcion}")

        # Paso 3: enviar el resultado de vuelta al modelo
        function_response_part = types.Part.from_function_response(
            name=nombre_funcion,
            response={"result": resultado_funcion},
            #id=function_id,
        )

        contents.append(response.candidates[0].content)
        contents.append(
            types.Content(role="user", parts=[function_response_part])
        )

        # Paso 4: obtener la siguiente respuesta del modelo
        response = client.models.generate_content(
            model=MODELO_LLM,
            contents=contents,
            config=config_agente,
        )

    # Extraer la respuesta final de texto
    respuesta_final = response.text
    print("-" * 60)
    print(f"Respuesta del Agente:\n")
    print(respuesta_final)
    return respuesta_final


print("Funcion ejecutar_consulta_agente lista.")

Funcion ejecutar_consulta_agente lista.


In [12]:
# Consulta de prueba 1
print("=" * 80)
print("CONSULTA DE PRUEBA 1")
print("=" * 80)

respuesta_1 = ejecutar_consulta_agente(
    "Cuales son los principales requisitos de seguridad para trabajos en minas subterraneas segun los documentos?"
)

CONSULTA DE PRUEBA 1
Pregunta: Cuales son los principales requisitos de seguridad para trabajos en minas subterraneas segun los documentos?
------------------------------------------------------------
  [Iteracion 1] El modelo solicita: buscar_informacion_tecnica({'query': 'requisitos de seguridad para trabajos en minas subterraneas'})
  [Iteracion 1] Funcion ejecutada. Resultado: 3015 chars
------------------------------------------------------------
Respuesta del Agente:

Según los documentos, los principales requisitos de seguridad para trabajos en minas subterráneas son los siguientes:

*   **Diseño y Protección de Entradas:** El acceso a la mina debe ser diseñado y protegido para prevenir el ingreso inadvertido y descontrolado de vehículos. Además, debe contar con una playa de estacionamiento subterránea con capacidad para un 20% más de vehículos y/o maquinarias para visitantes y emergencias. (Fuente 1: rsso-2024.pdf, página 183)
*   **Medios Seguros de Entrada y Escape:** Debe di

In [13]:
# Consulta de prueba 2
print("=" * 80)
print("CONSULTA DE PRUEBA 2")
print("=" * 80)

respuesta_2 = ejecutar_consulta_agente(
    "Que establece la normativa sobre el manejo de residuos y relaves mineros?"
)

CONSULTA DE PRUEBA 2
Pregunta: Que establece la normativa sobre el manejo de residuos y relaves mineros?
------------------------------------------------------------
  [Iteracion 1] El modelo solicita: buscar_informacion_tecnica({'query': 'normativa sobre manejo de residuos y relaves mineros'})
  [Iteracion 1] Funcion ejecutada. Resultado: 3034 chars
------------------------------------------------------------
Respuesta del Agente:

La normativa sobre el manejo de residuos y relaves mineros establece lo siguiente:

*   **Residuos Sólidos:** El manejo de los residuos sólidos de origen doméstico e industrial generados en la unidad minera debe realizarse de acuerdo con la Ley Nº 27314, Ley General de Residuos Sólidos, sus modificatorias o la norma que la sustituya, y demás normas vigentes aplicables. Además, debe estar en concordancia con el Programa Anual de Seguridad y Salud Ocupacional elaborado por el titular de la actividad minera. (Fuente: rsso-2024.pdf, página 188)
*   **Residuos M

In [14]:
# Consulta de prueba 3
print("=" * 80)
print("CONSULTA DE PRUEBA 3")
print("=" * 80)

respuesta_3 = ejecutar_consulta_agente(
    "Cuales son los Instrumentos de gestión aprobados para la minera las bambas?"
)

CONSULTA DE PRUEBA 3
Pregunta: Cuales son los Instrumentos de gestión aprobados para la minera las bambas?
------------------------------------------------------------
  [Iteracion 1] El modelo solicita: buscar_informacion_tecnica({'query': 'Instrumentos de gestión aprobados para la minera las bambas'})
  [Iteracion 1] Funcion ejecutada. Resultado: 3120 chars
------------------------------------------------------------
Respuesta del Agente:

Los instrumentos de gestión aprobados para la minera Las Bambas son los siguientes:

*   **EIA de la U.M. Las Bambas:** R.D. N° 073-2011-MEM/AAM, aprobado el 07 de marzo de 2011.
*   **Primera Modificación del EIA (Primera MEIA):** R.D. N° 305-2013-MEM-AAM, aprobado el 14 de agosto de 2013.
*   **Informe Técnico Sustentatorio (Primer ITS):** R.D. N° 319-2013-MEM-AAM, aprobado el 17 de septiembre de 2013.
*   **Informe Técnico Sustentatorio (Segundo ITS):** R.D. N° 078-2014-MEM-DGAAM, aprobado el 13 de febrero de 2014.
*   **Segunda Modificación del

In [16]:
# Guardar resultados en un archivo para revision
with open("resultados_poc.md", "w", encoding="utf-8") as f:
    f.write("# Resultados del PoC RAG Agentico\n\n")
    f.write(f"Modelo LLM: {MODELO_LLM}\n")
    f.write(f"Modelo Embedding: {MODELO_EMBEDDING}\n")
    f.write(f"Total chunks indexados: {coleccion.count()}\n")
    f.write(f"Top-K: {TOP_K}\n\n")
    f.write("## Consulta 1\n\n")
    f.write("**Pregunta:** Cuales son los principales requisitos de seguridad "
            "para trabajos en minas subterraneas segun los documentos?\n\n")
    f.write(f"**Respuesta:**\n\n{respuesta_1}\n\n")
    f.write("## Consulta 2\n\n")
    f.write("**Pregunta:** Que establece la normativa sobre el manejo de "
            "residuos y relaves mineros?\n\n")
    f.write(f"**Respuesta:**\n\n{respuesta_2}\n\n")
    f.write("## Consulta 3\n\n")
    f.write("**Pregunta:** Cuales son los Instrumentos de gestión aprobados para la minera las bambas?\n\n")
    f.write(f"**Respuesta:**\n\n{respuesta_3}\n\n")

print("Resultados guardados en 'resultados_poc.md'.")
print("\nPoC RAG Agentico ejecutado exitosamente.")

Resultados guardados en 'resultados_poc.md'.

PoC RAG Agentico ejecutado exitosamente.
